# «ПаркУчёт»: модели и диаграммы на SysML v2

Ноутбук для ядра Jupyter SysML (SysML v2 Pilot Implementation 2026-09.1, ядро 0.63.1).
Первые две ячейки загружают модели `parking-system.sysml` и `ui-mockups.sysml`,
остальные строят рисунки 1 – 7 спецификации командой `%viz`.

In [ ]:
// Модель системы «ПаркУчёт» на языке SysML v2 (текстовая нотация).
// Диаграммы: images/use-cases.png, images/exit-flow.png, images/session-states.png
package ParkUchet {
    private import ScalarValues::*;

    // Участник и система
    part def Operator;

    part def ParkingSystem {
        attribute capacity : Integer;
        attribute hourlyRate : Real;
        attribute freeMinutes : Integer;
    }

    // Сессия стоянки
    item def ParkingSession {
        attribute plate : String;
        attribute brand : String;
        attribute entryTime : String;
        attribute exitTime : String;
        attribute cost : Real;
    }

    // Варианты использования (рисунок use-cases.png)
    use case 'Работа оператора' {
        subject 'ПаркУчёт' : ParkingSystem;
        actor 'Оператор' : Operator;

        include use case 'Регистрация въезда';
        include use case 'Оформление выезда' {
            include use case 'Расчёт стоимости';
        }
        include use case 'Просмотр свободных мест';
        include use case 'Просмотр истории';
        include use case 'Настройка тарифа и вместимости';
    }

    // Оформление выезда (рисунок exit-flow.png)
    action def 'Оформить выезд' {
        attribute timeValid : Boolean;
        attribute confirmed : Boolean;

        first start;
        then action 'Выбрать автомобиль';
        then merge 'Ввод времени';
        then action 'Указать время выезда';
        then decide 'Проверка времени';
            if timeValid then 'Рассчитать стоимость';
            else 'Показать ошибку времени';

        action 'Показать ошибку времени';
        then 'Ввод времени';

        action 'Рассчитать стоимость';
        then action 'Показать стоимость';
        then decide 'Проверка оплаты';
            if confirmed then 'Завершить сессию';
            else done;

        action 'Завершить сессию';
        then action 'Обновить свободные места';
        then done;
    }

    // Состояния сессии стоянки (рисунок session-states.png)
    attribute def 'Въезд';
    attribute def 'Начат выезд';
    attribute def 'Отмена';
    attribute def 'Оплата принята';

    state def 'Состояния сессии' {
        entry; then 'Новая';

        state 'Новая';
        accept 'Въезд' then 'На стоянке';

        state 'На стоянке';
        accept 'Начат выезд' then 'Расчёт стоимости';

        state 'Расчёт стоимости';
        transition first 'Расчёт стоимости' accept 'Отмена' then 'На стоянке';
        transition first 'Расчёт стоимости' accept 'Оплата принята' then 'Завершена';

        state 'Завершена';
    }

    // Основные требования к расчёту и учёту мест (полный перечень в srs/srs.md)
    requirement def <'REQ-FUNC-005'> 'Запрет въезда при отсутствии мест' {
        doc /* При нуле свободных мест въезд отклоняется. */
    }

    requirement def <'REQ-FUNC-007'> 'Правило расчёта стоимости' {
        doc /* Стоимость рассчитывается по тарифу на момент въезда. */
    }

    requirement def <'REQ-FUNC-009'> 'Показ свободных мест' {
        doc /* Свободные места равны вместимости минус занятые места. */
    }
}

In [ ]:
// Макеты окон «ПаркУчёт» на языке SysML v2.
// Окно и группа элементов – части (part), элементы окна – вложенные части;
// имя части – надпись, которую видит оператор, определение – вид элемента.
// Изображения: images/mockup-main.png, mockup-exit.png, mockup-history.png, mockup-settings.png
package ParkUchetUi {

    // Виды элементов интерфейса
    part def Window;
    part def Group;
    part def Label;
    part def TextField;
    part def Button;
    part def Table;

    // Рисунок mockup-main.png
    part 'Главное окно «ПаркУчёт»' : Window {
        part 'Всего мест, занято, свободно' : Label;
        part 'Въезд автомобиля' : Group {
            part 'Госномер' : TextField;
            part 'Марка' : TextField;
            part 'Время въезда' : TextField;
            part 'Зарегистрировать въезд' : Button;
        }
        part 'Автомобили на стоянке' : Group {
            part 'Госномер, марка, въезд, длительность' : Table;
        }
        part 'Действия' : Group {
            part 'Оформить выезд' : Button;
            part 'История' : Button;
            part 'Настройки' : Button;
        }
    }

    // Рисунок mockup-exit.png
    part 'Окно «Оформление выезда»' : Window {
        part 'Госномер, марка, время въезда' : Label;
        part 'Выезд' : Group {
            part 'Время выезда' : TextField;
            part 'Длительность' : Label;
            part 'Тариф' : Label;
            part 'К оплате, BYN' : Label;
        }
        part 'Кнопки' : Group {
            part 'Оплата принята' : Button;
            part 'Отмена' : Button;
        }
    }

    // Рисунок mockup-history.png
    part 'Окно «История стоянок»' : Window {
        part 'Госномер, въезд, выезд, сумма' : Table;
        part 'Закрыть' : Button;
    }

    // Рисунок mockup-settings.png
    part 'Окно «Настройки»' : Window {
        part 'Параметры' : Group {
            part 'Вместимость, мест' : TextField;
            part 'Ставка за час, BYN' : TextField;
            part 'Льготный период, мин' : TextField;
        }
        part 'Кнопки' : Group {
            part 'Сохранить' : Button;
            part 'Отмена' : Button;
        }
    }
}

In [ ]:
%viz --view=Case --style=LR ParkUchet::'Работа оператора'

In [ ]:
%viz --view=Action ParkUchet::'Оформить выезд'

In [ ]:
%viz --view=State ParkUchet::'Состояния сессии'

In [ ]:
%viz --view=Interconnection --style=LR ParkUchetUi::'Главное окно «ПаркУчёт»'

In [ ]:
%viz --view=Interconnection --style=LR ParkUchetUi::'Окно «Оформление выезда»'

In [ ]:
%viz --view=Interconnection --style=LR ParkUchetUi::'Окно «История стоянок»'

In [ ]:
%viz --view=Interconnection --style=LR ParkUchetUi::'Окно «Настройки»'